# 01 — RAG Foundations: Evidence Before Answers

**Track:** Beginner · **Stage:** Foundation

NovaTech wants an Enterprise Knowledge Assistant that can answer questions over finance reviews, HR policies, IT runbooks, project docs, and vendor contracts. In this first notebook, we build the basic Retrieval-Augmented Generation (RAG) loop using **LangChain**, one of the most prominent enterprise AI SDKs.

## What you will build

- A basic LangChain RAG pipeline.
- A demonstration of naive generation vs. grounded generation.
- An inspectable trace of retrieved evidence.
- An understanding of factual grounding, provenance, and citations.

## Setup: Model and SDK

For this module, we use LangChain. We will use a mock LLM for deterministic, zero-cost local execution. 

> **Disclaimer:** The `FakeListLLM` used here only demonstrates that the pipeline is wired correctly (retrieval → context → prompt → generator interface). It does **not** prove that a real LLM will follow grounding instructions, prevent hallucinations, or cite sources correctly. Real evaluation is required for that.

In [1]:
# !pip install langchain langchain-core

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import FakeListLLM

## 1. The Naive Approach (No RAG)

First, let's ask a question without providing any external context. The model will either hallucinate or state it doesn't know.

In [2]:
# We use a FakeListLLM to simulate an LLM's hallucination for this exercise.
# In production, you would use: from langchain_openai import ChatOpenAI; llm = ChatOpenAI()
mock_hallucinated_responses = [
    "NovaTech's Q2 2025 revenue increased by 5%."
]
llm = FakeListLLM(responses=mock_hallucinated_responses)

question = "What increased by 14% at NovaTech in Q2 2025?"
print("Model answer without RAG:", llm.invoke(question))

Model answer without RAG: NovaTech's Q2 2025 revenue increased by 5%.


The model gives a plausible but incorrect answer (hallucination). RAG fixes this by enforcing a contract: **the model may only answer using evidence we provide.**

## 2. Ingesting Enterprise Data

Let's define some mock internal documents. Notice that we include rich metadata. 

Not all metadata should be sent to the LLM. 
- **Generator-useful metadata:** document title, section, effective date.
- **Pipeline/operational metadata:** chunk ID, document ID, source file, tenant, ACLs (used for retrieval filtering, not LLM context).

In [3]:
documents = [
    Document(
        page_content="NovaTech's Q2 2025 financial review: Cloud infrastructure costs increased by 14% due to the new cluster deployment.", 
        metadata={
            "document_id": "fin-2025-q2",
            "chunk_id": "fin-2025-q2#infra-01",
            "title": "Q2 2025 Financial Review",
            "section": "Infrastructure",
            "source": "finance_review_q2.md", 
            "tenant": "finance",
            "version": "1.0"
        }
    ),
    Document(
        page_content="HR Policy: Parental leave has been extended to 16 weeks globally.", 
        metadata={
            "document_id": "hr-pol-04",
            "chunk_id": "hr-pol-04#benefits-02",
            "title": "Global HR Policies",
            "section": "Benefits",
            "source": "hr_policy_v2.md", 
            "tenant": "hr",
            "version": "2.0"
        }
    ),
    Document(
        page_content="Runbook 17: For checkout errors, correlate the 08:42 deployment with payment dependency latency before proposing rollback.", 
        metadata={
            "document_id": "rb-17",
            "chunk_id": "rb-17#checkout-01",
            "title": "Runbook 17: Checkout",
            "section": "Troubleshooting",
            "source": "runbooks/checkout.md", 
            "tenant": "engineering",
            "version": "1.2"
        }
    )
]

def teaching_retriever(query: str):
    """
    Teaching retriever — deliberately simple and deterministic. 
    Course 02 replaces this with real semantic retrieval.
    """
    if "14%" in query or "finance" in query.lower():
        return [documents[0]]
    elif "leave" in query.lower() or "hr" in query.lower():
        return [documents[1]]
    else:
        return [documents[2]]

# Test the mock retriever
candidates = teaching_retriever(question)
print(f"Retrieved {len(candidates)} document(s).")


Retrieved 1 document(s).


## 3. Context Construction and Stable IDs

We need to format the retrieved `candidates` into a string `evidence` that the LLM can read. We will assign stable local evidence IDs (e.g., `[E1]`) to each chunk. This allows the model to cite `[E1]`, which the application can later map back to the durable `chunk_id` and `source`.

In [4]:
evidence_mapping = {}

def format_docs(docs):
    global evidence_mapping
    evidence_mapping.clear()
    
    formatted_chunks = []
    for i, d in enumerate(docs, start=1):
        evidence_id = f"E{i}"
        evidence_mapping[evidence_id] = d.metadata
        
        chunk = (
            f"<EVIDENCE id=\"{evidence_id}\">\n"
            f"Title: {d.metadata['title']}\n"
            f"Section: {d.metadata['section']}\n"
            f"Content:\n{d.page_content}\n"
            f"</EVIDENCE>"
        )
        formatted_chunks.append(chunk)
    return "\n\n".join(formatted_chunks)

## 4. The Grounded RAG Pipeline

Now we define a strict prompt that instructs the model to use the evidence, cite the IDs, and abstain if necessary.

In [5]:
template = """
You are a helpful NovaTech enterprise assistant.
Answer the question based ONLY on the following context.
If you cannot answer the question based on the context, say "I do not know based on the provided evidence."
If the supplied evidence conflicts, report the conflict rather than inventing a resolution.
Cite the evidence IDs (e.g., [E1]) for your factual claims.

Context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Update our fake LLM to simulate the correct grounded response
llm = FakeListLLM(responses=["Based on the financial review, cloud infrastructure costs increased by 14% due to the new cluster deployment [E1]."])

rag_chain = (
    {"context": RunnableLambda(teaching_retriever) | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("--- Actual Prompt Sent to LLM ---")
# Let's peek at the actual prompt we built
evidence_str = format_docs(candidates)
print(prompt.format(context=evidence_str, question=question))
print("---------------------------------\n")

print("Model answer WITH RAG:")
answer = rag_chain.invoke(question)
print(answer)


--- Actual Prompt Sent to LLM ---
Human: 
You are a helpful NovaTech enterprise assistant.
Answer the question based ONLY on the following context.
If you cannot answer the question based on the context, say "I do not know based on the provided evidence."
If the supplied evidence conflicts, report the conflict rather than inventing a resolution.
Cite the evidence IDs (e.g., [E1]) for your factual claims.

Context:
<EVIDENCE id="E1">
Title: Q2 2025 Financial Review
Section: Infrastructure
Content:
NovaTech's Q2 2025 financial review: Cloud infrastructure costs increased by 14% due to the new cluster deployment.
</EVIDENCE>

Question: What increased by 14% at NovaTech in Q2 2025?

---------------------------------

Model answer WITH RAG:
Based on the financial review, cloud infrastructure costs increased by 14% due to the new cluster deployment [E1].


## 5. Provenance and Conflict Experiment

What happens if policies change? Semantic metadata (like `version` or `effective_date`) helps interpretation and conflict resolution. Let's see how passing semantic metadata helps.

In [6]:
conflicting_documents = [
    Document(
        page_content="Tier 1 may approve failovers.",
        metadata={"title": "Escalation Policy", "section": "Permissions", "version": "2024", "effective_date": "2024-01-01", "document_id": "pol-1-v1"}
    ),
    Document(
        page_content="Tier 2 must approve failovers.",
        metadata={"title": "Escalation Policy", "section": "Permissions", "version": "2026", "effective_date": "2026-01-01", "document_id": "pol-1-v2"}
    )
]

print("--- Formatted Context with Conflict ---")
print(format_docs(conflicting_documents))


--- Formatted Context with Conflict ---
<EVIDENCE id="E1">
Title: Escalation Policy
Section: Permissions
Content:
Tier 1 may approve failovers.
</EVIDENCE>

<EVIDENCE id="E2">
Title: Escalation Policy
Section: Permissions
Content:
Tier 2 must approve failovers.
</EVIDENCE>


With this context, a properly instructed LLM can report the conflict ("The 2024 policy says Tier 1, but the 2026 policy requires Tier 2") instead of silently collapsing them.

## Reflection

Consider these four levels of context formatting:

**A.** Content only
**B.** Content + filename (`policy.md`)
**C.** Content + Evidence ID + Title + Section
**D.** Content + Evidence ID + Title + Section + Version/Date

Ask yourself:
1. Does the factual answer change between these? (Factual grounding)
2. Can the answer be traced to its source? (Provenance & Citation)
3. Can conflicting documents be distinguished? (Generator-visible metadata)
4. Which metadata actually helps the LLM interpret the text, and which only helps operational traceability (like Tenant ID)?

In the next lab, we will replace the `teaching_retriever` with a real local vector database and embedding model.